# Kuza AI — An East African Agricultural Assistant

In [1]:
## 1. Setup and Installation
!pip install unsloth unsloth_zoo
!pip uninstall -y fbgemm-gpu fbgemm-gpu-genai torchao
!pip install --pre torchao fbgemm-gpu fbgemm-gpu-genai --index-url https://download.pytorch.org/whl/nightly/cu126
!pip install trl peft accelerate xformers

import os
import torch
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only, get_chat_template
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from huggingface_hub import login

login(token="my_hf_token")
SEED = 42

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/2

[root|ERROR]Could not load the library 'fbgemm_gpu_config.so'!


Could not load this library: /usr/local/lib/python3.12/dist-packages/fbgemm_gpu/fbgemm_gpu_config.so



[root|ERROR]Could not load the library 'fbgemm_gpu_config.so'!


Could not load this library: /usr/local/lib/python3.12/dist-packages/fbgemm_gpu/fbgemm_gpu_config.so



[root|ERROR]Could not load the library 'fbgemm_gpu_config.so'!


Could not load this library: /usr/local/lib/python3.12/dist-packages/fbgemm_gpu/fbgemm_gpu_config.so



[root|ERROR]Could not load the library 'fbgemm_gpu_config.so'!


Could not load this library: /usr/local/lib/python3.12/dist-packages/fbgemm_gpu/fbgemm_gpu_config.so





🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# 1. Load Model and Tokenizer
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it-qat-q4_0-unquantized",
    max_seq_length=512,
    dtype=None,
    load_in_4bit=False,
    load_in_16bit=True,
)

# 2. Apply PEFT (RsLoRA) with QAT
model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    full_finetuning=False,

    finetune_vision_layers=False,    # Text-only fine-tune
    finetune_language_layers=True,   # Target language layers
    finetune_attention_modules=True, # Target attention
    finetune_mlp_modules=True,       # Target MLPs

    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    qat_scheme="int4",
    use_rslora=True,       # Enables alpha / sqrt(r) scaling
    use_dora=False,        # Explicitly disabled to save throughput
)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.8.19: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Gemma4ForConditionalGeneration LOAD REPORT from: unsloth/gemma-4-E2B-it-qat-q4_0-unquantized
Key                                                           | Status  | 
--------------------------------------------------------------+---------+-
model.language_model.layers.{15...34}.self_attn.k_norm.weight | MISSING | 
model.language_model.layers.{15...34}.self_attn.k_proj.weight | MISSING | 
model.language_model.layers.{15...34}.self_attn.v_proj.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Applying QAT to mitigate quantization degradation
trainable params: 50,675,712 || all params: 5,173,853,728 || trainable%: 0.9795


In [3]:
import os
import json
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
MAIN_DATASET_REPO = "kuzaai/agri_sft_prod_dedup_25k"
SWAHILI_DATASET_REPO = "kuzaai/agri_sft_25k_swahili"
NO_ROBOTS_FRACTION = 0.10      # ~10% of main train size
ADVERSARIAL_FRACTION = 0.05    # small auxiliary mix, keeps Kuza dominant
SWAHILI_FRACTION = 0.15

SYSTEM_PROMPT = (
    "You are Kuza, an East African agricultural assistant. "
    "Be direct. Use exact numbers when known. Avoid vague phrasing. "
    "If uncertain, say so briefly. "
    "Respond in the same language as the query.\n"
    "Wewe ni Kuza, msaidizi wa kilimo wa Afrika Mashariki. "
    "Kuwa wa moja kwa moja. Tumia namba kamili unapojua. "
    "Epuka maneno ya kawaida. Kama huna uhakika, sema kwa ufupi. "
    "Jibu kwa lugha ya swali."
)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def keep_valid(example):
    """Keeps only rows with both instruction and response present."""
    return (
        example.get("instruction") is not None and
        example.get("response") is not None and
        str(example["instruction"]).strip() != "" and
        str(example["response"]).strip() != ""
    )


def get_primary_split(ds_dict):
    """
    Picks the most likely main split from a DatasetDict.
    Prefers 'train' if present, otherwise uses the first available split.
    """
    if "train" in ds_dict:
        return ds_dict["train"]
    return ds_dict[list(ds_dict.keys())[0]]


def format_main_chat(example):
    """Main cleaned agri dataset - English."""
    messages = example.get("messages", [])
    if len(messages) >= 2:
        first, second = messages[0], messages[1]
        if first.get("role") == "user" and second.get("role") == "assistant":
            return {
                "instruction": first.get("content", "") or "",
                "response": second.get("content", "") or "",
                "language": "english"
            }
    return {"instruction": "", "response": "", "language": "english"}


def format_swahili_chat(example):
    """Swahili dataset - direct translation."""
    messages = example.get("messages", [])
    if len(messages) >= 2:
        first, second = messages[0], messages[1]
        if first.get("role") == "user" and second.get("role") == "assistant":
            return {
                "instruction": first.get("content", "") or "",
                "response": second.get("content", "") or "",
                "language": "swahili"  # kept for mix reporting only
            }
    return {"instruction": "", "response": "", "language": "swahili"}


def format_adv(example):
    """Formats adversarial examples - English only."""
    if "conversations" in example and len(example["conversations"]) >= 2:
        convs = example["conversations"]
        if convs[0].get("from") == "human" and convs[1].get("from") == "gpt":
            return {
                "instruction": convs[0].get("value", "") or "",
                "response": convs[1].get("value", "") or "",
                "language": "english"
            }
    if "messages" in example and len(example["messages"]) >= 2:
        msgs = example["messages"]
        if msgs[0].get("role") == "user" and msgs[1].get("role") == "assistant":
            return {
                "instruction": msgs[0].get("content", "") or "",
                "response": msgs[1].get("content", "") or "",
                "language": "english"
            }
    return {"instruction": "", "response": "", "language": "english"}


def format_general(example):
    """Formats the No Robots dataset - English only."""
    messages = example.get("messages", [])
    if len(messages) >= 2:
        first, second = messages[0], messages[1]
        if first.get("role") == "user" and second.get("role") == "assistant":
            instruction = first.get("content", "") or ""
            response = second.get("content", "") or ""

            if len(instruction.split()) > 200 or len(response.split()) > 250:
                return {"instruction": "", "response": "", "language": "english"}

            return {
                "instruction": instruction,
                "response": response,
                "language": "english"
            }
    return {"instruction": "", "response": "", "language": "english"}


def sample_to_ratio(ds, target_size, seed=SEED):
    """Downsample a dataset to target_size if needed."""
    if ds is None or len(ds) == 0:
        return ds
    if len(ds) <= target_size:
        return ds
    return ds.shuffle(seed=seed).select(range(target_size))


# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

# Load English main dataset
ds_main_dict = load_dataset(MAIN_DATASET_REPO)
ds_main = get_primary_split(ds_main_dict)
ds_main = ds_main.map(format_main_chat, remove_columns=ds_main.column_names)
ds_main = ds_main.filter(keep_valid)

# Load Swahili dataset
ds_swahili_dict = load_dataset(SWAHILI_DATASET_REPO)
ds_swahili = get_primary_split(ds_swahili_dict)
ds_swahili = ds_swahili.map(format_swahili_chat, remove_columns=ds_swahili.column_names)
ds_swahili = ds_swahili.filter(keep_valid)

# Split both datasets
split_main = ds_main.train_test_split(test_size=0.05, seed=SEED)
split_swahili = ds_swahili.train_test_split(test_size=0.05, seed=SEED)

ds_main_train = split_main["train"]
ds_main_val = split_main["test"]
ds_swahili_train = split_swahili["train"]
ds_swahili_val = split_swahili["test"]

# ------------------------------------------------------------
# 2. Load adversarial examples (English only)
# ------------------------------------------------------------

try:
    ds_adv_dict = load_dataset("kuzaai/adversarial_examples")
    ds_adv = get_primary_split(ds_adv_dict)
    ds_adv = ds_adv.map(format_adv, remove_columns=ds_adv.column_names)
    ds_adv = ds_adv.filter(keep_valid)

    if len(ds_adv) > 0:
        adv_target = max(1, int(len(ds_main_train) * ADVERSARIAL_FRACTION))
        ds_adv = sample_to_ratio(ds_adv, adv_target, seed=SEED)

        adv_split = ds_adv.train_test_split(test_size=0.10, seed=SEED)
        ds_adv_train = adv_split["train"]
        ds_adv_eval = adv_split["test"]
    else:
        print("Warning: Adversarial dataset was empty after formatting.")
        ds_adv_train = None
        ds_adv_eval = None
except Exception as e:
    print(f"Adversarial dataset loading failed: {e}")
    ds_adv_train = None
    ds_adv_eval = None

# ------------------------------------------------------------
# 3. Load No Robots (English only)
# ------------------------------------------------------------

ds_general = load_dataset("HuggingFaceH4/no_robots", split="train")
ds_general = ds_general.map(format_general, remove_columns=ds_general.column_names)
ds_general = ds_general.filter(keep_valid)

# Downsample no_robots
general_target = max(1, int(len(ds_main_train) * NO_ROBOTS_FRACTION))
ds_general = sample_to_ratio(ds_general, general_target, seed=SEED)

# ------------------------------------------------------------
# 4. Combine training splits with Swahili
# ------------------------------------------------------------

# Calculate Swahili target size based on fraction
swahili_target = max(1, int(len(ds_main_train) * SWAHILI_FRACTION))
ds_swahili_train = sample_to_ratio(ds_swahili_train, swahili_target, seed=SEED)

train_parts = [ds_main_train, ds_general, ds_swahili_train]

if ds_adv_train is not None and len(ds_adv_train) > 0:
    train_parts.append(ds_adv_train)

train_dataset = concatenate_datasets(train_parts).shuffle(seed=SEED)

# ------------------------------------------------------------
# 5. Combine evaluation splits with Swahili
# ------------------------------------------------------------

eval_parts = [ds_main_val, ds_swahili_val]

if ds_adv_eval is not None and len(ds_adv_eval) > 0:
    eval_parts.append(ds_adv_eval)

eval_dataset = concatenate_datasets(eval_parts).shuffle(seed=SEED)

# ------------------------------------------------------------
# 6. Format to chat template
# ------------------------------------------------------------

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def formatting_prompts_func(examples):
    texts = []
    for inst, resp in zip(examples["instruction"], examples["response"]):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": inst},
            {"role": "assistant", "content": resp},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

PROMPT_OUT = "/kaggle/working/kuza_system_prompt.json"
os.makedirs("/kaggle/working", exist_ok=True)
with open(PROMPT_OUT, "w", encoding="utf-8") as f:
    json.dump({"system_prompt": SYSTEM_PROMPT}, f, ensure_ascii=False, indent=2)

print(f"✅ Saved system prompt to {PROMPT_OUT}")

print(f"English train size: {len(ds_main_train)}")
print(f"Swahili train size: {len(ds_swahili_train)}")
print(f"English val size: {len(ds_main_val)}")
print(f"Swahili val size: {len(ds_swahili_val)}")
print(f"Final train dataset size: {len(train_dataset)}")
print(f"Final eval dataset size: {len(eval_dataset)}")

# Quick sanity check on one rendered sample
print("\n--- Sample rendered training text (first 600 chars) ---")
print(train_dataset["text"][0][:600])
print("--------------------------------------------------------")

gemma4_agri_sft_25k.jsonl:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25506 [00:00<?, ? examples/s]

Map:   0%|          | 0/25506 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25506 [00:00<?, ? examples/s]

gemma4_agri_sft_25k_swahili.jsonl:   0%|          | 0.00/34.2M [00:00<?, ?B/s]

...

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9500 [00:00<?, ? examples/s]

Map:   0%|          | 0/30467 [00:00<?, ? examples/s]

Map:   0%|          | 0/2561 [00:00<?, ? examples/s]

✅ Saved system prompt to /kaggle/working/kuza_system_prompt.json
English train size: 24230
Swahili train size: 3634
English val size: 1276
Swahili val size: 1265
Final train dataset size: 30467
Final eval dataset size: 2561

--- Sample rendered training text (first 600 chars) ---
<bos><|turn>system
You are Kuza, an East African agricultural assistant. Be direct. Use exact numbers when known. Avoid vague phrasing. If uncertain, say so briefly. Respond in the same language as the query.
Wewe ni Kuza, msaidizi wa kilimo wa Afrika Mashariki. Kuwa wa moja kwa moja. Tumia namba kamili unapojua. Epuka maneno ya kawaida. Kama huna uhakika, sema kwa ufupi. Jibu kwa lugha ya swali.<turn|>
<|turn>user
When should I plan to harvest beans and maize in Chebilat to ensure the best yield?<turn|>
<|turn>model
Harvest beans in Chebilat around September October and maize around October N
--------------------------------------------------------


In [4]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        optim="adamw_torch_fused",
        lr_scheduler_type="constant_with_warmup",
        max_grad_norm=1.0,
        max_seq_length=512,
        logging_steps=50,
        report_to="none",
        run_name="gemma4-edge-agri-swahili-25k",
        eval_strategy="steps",
        save_strategy="steps",
        eval_steps=500,
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        output_dir="/kaggle/working/agri-lora-swahili-adapter",
        neftune_noise_alpha=5.0,
    ),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=1e-3,
        )
    ],
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/30467 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2561 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/30467 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/30467 [00:00<?, ? examples/s]

Unsloth: Removed 7 out of 30467 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map (num_proc=2):   0%|          | 0/2561 [00:00<?, ? examples/s]

In [5]:
trainer_stats = trainer.train()

output_dir = "/kaggle/working/agri-lora-swahili-adapter"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Training complete. LoRA adapter with Swahili support saved to {output_dir}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,460 | Num Epochs = 2 | Total steps = 1,904
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 50,675,712 of 5,173,853,728 (0.98% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
500,1.515690,1.509637
1000,1.377466,1.430062
1500,1.360493,1.370693
1904,1.323325,1.340555


Filter:   0%|          | 0/2561 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/agri-lora-swahili-adapter/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/agri-lora-swahili-adapter/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/agri-lora-swahili-adapter/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/agri-lora-swahili-adapter/checkpoint-1904/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/agri-lora-swahili-adapter/tokenizer_config.json.


✅ Training complete. LoRA adapter with Swahili support saved to /kaggle/working/agri-lora-swahili-adapter
